# Volume 6, Chapter 8: Product Measures and Change of Variables (Python companion)

Exercises first, solutions at the end. Products and changes of variables on a computer: sampling uniformly in a high-dimensional ball (rejection against the Gaussian method), the Box–Muller transform and Gaussian vectors with a given covariance, the curse of dimension for nearest-neighbour search, the failure of Fubini without integrability, and areas of images under linear and nonlinear maps. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P8.1 (★) CS application: sampling uniformly in a ball.** (a) Rejection sampling: draw points uniformly in the cube $[-1,1]^d$ and keep those in the unit ball. Measure the acceptance rate for $d=2,5,10,15,20$ and compare with $\lambda_d(B^d)/2^d$ (Theorem 8.11). How many cube points would one ball point cost in dimension $50$? (b) The Gaussian method of Exercise 8.5: $U^{1/d}Z/|Z|$. Check uniformity in $d=20$ by testing that $|X|^d$ is uniform on $[0,1]$ and that $X_1\sqrt d$ is approximately standard Gaussian (Exercise 8.7).

**P8.2 (★★) The Box–Muller transform.** Let $R=\sqrt{-2\ln U_1}$ and $\Theta=2\pi U_2$ for independent uniforms. Show with polar coordinates that $(R\cos\Theta,R\sin\Theta)$ is a standard Gaussian vector of $\R^2$, check it by simulation, and compare its speed with numpy's generator. Then simulate a Gaussian vector with covariance $\Sigma=\begin{pmatrix}2&1\\1&1\end{pmatrix}$ through $X=AZ$ with $A$ the Cholesky factor, and compare the empirical density with Theorem 8.12.

**P8.3 (★★) CS application: the curse of dimension for nearest neighbours.** Draw $1000$ uniform points in $[0,1]^d$ and a query point. Compute the ratio between the largest and the smallest distance from the query to the points, for $d=2,10,100,1000$. Explain with Exercise 8.7 and the law of large numbers why the ratio tends to $1$, and what that means for nearest-neighbour search.

**P8.4 (★) When Fubini fails.** Compute numerically the two iterated integrals of $f(x,y)=\frac{x^2-y^2}{(x^2+y^2)^2}$ on $]0,1[^2$ (Exercise 8.1) and the two iterated sums of the matrix of the "Pause and think" box after Theorem 8.6, truncated to $N\times N$. Where does the discrepancy live?

**P8.5 (★) Volumes under maps.** Estimate $\lambda_2(T([0,1]^2))$ by Monte Carlo for a few matrices $T$ and compare with $|\det T|$. Then estimate the area of the image of $[0,1]^2$ under $\varphi(x,y)=(x+\frac14\sin\frac{3y}2,\ y+0.15x^2)$ (the map of Figure 8.3), both by Monte Carlo in the image and by $\int_{[0,1]^2}|J_\varphi|$.

## Solutions

### P8.1

In [ ]:
import numpy as np
from math import gamma, pi
from scipy.stats import kstest
rng = np.random.default_rng(0)
omega = lambda d: pi ** (d / 2) / gamma(d / 2 + 1)
for d in [2, 5, 10, 15, 20]:
    N = 2_000_000
    acc = 0
    for _ in range(5):
        X = rng.uniform(-1, 1, (N // 5, d))
        acc += np.count_nonzero(np.einsum("ij,ij->i", X, X) <= 1)
    print(f"d={d:2d}: acceptance {acc/N:.2e}   omega_d/2^d = {omega(d)/2**d:.2e}")
print(f"d=50: one ball point costs about {2**50/omega(50):.1e} cube points")

d, n = 20, 100_000
Z = rng.standard_normal((n, d)); U = rng.random(n)
X = U[:, None] ** (1 / d) * Z / np.linalg.norm(Z, axis=1)[:, None]
r = np.linalg.norm(X, axis=1)
print("max |X| =", r.max())
print("KS test of |X|^d against uniform: p =", kstest(r ** d, "uniform").pvalue)
print("X_1 sqrt(d): mean %.3f, variance %.3f (Gaussian limit: 0, 1)" % ((X[:, 0] * np.sqrt(d)).mean(), (X[:, 0] * np.sqrt(d)).var()))

The acceptance rate is $\lambda_d(B^d)/2^d$: about $0.79$ in the plane, $0.0025$ in dimension $10$, $2.5\cdot10^{-8}$ in dimension $20$ (with $2\cdot10^6$ trials one sees about $0$ accepted points), and in dimension $50$ one would need about $10^{28}$ cube points per ball point. Rejection sampling is the curse of dimension in its purest form: the ball is a vanishing part of its bounding cube. The Gaussian method costs $d+1$ random numbers per point in every dimension. It is exact (Exercise 8.5): $|X|^d$ passes the uniformity test, because $\mathbb P(|X|\le r)=r^d$, and $X_1\sqrt d$ has mean $0$ and variance $\frac d{d+2}\approx0.91$, close to the Gaussian limit of Exercise 8.7 (the exact variance of $X_1$ is $\frac1{d+2}$).

### P8.2

In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt
from scipy.stats import kstest
rng = np.random.default_rng(1)
def box_muller(n):
    u1, u2 = rng.random(n), rng.random(n)
    r = np.sqrt(-2 * np.log1p(-u1)); th = 2 * np.pi * u2         # log1p(-u) avoids log(0)
    return r * np.cos(th), r * np.sin(th)
x, y = box_muller(10 ** 6)
print("KS p-values for x and y against N(0,1):", kstest(x, "norm").pvalue, kstest(y, "norm").pvalue)
print("correlation of x and y:", np.corrcoef(x, y)[0, 1])
t = time.perf_counter(); box_muller(10 ** 7); t1 = time.perf_counter() - t
t = time.perf_counter(); rng.standard_normal(2 * 10 ** 7); t2 = time.perf_counter() - t
print(f"time for 2e7 Gaussians: Box-Muller {t1:.2f}s, numpy (ziggurat) {t2:.2f}s")

Sigma = np.array([[2.0, 1.0], [1.0, 1.0]]); A = np.linalg.cholesky(Sigma)
Z = rng.standard_normal((2, 200_000)); X = A @ Z
print("empirical covariance:\n", np.cov(X))
H, xe, ye = np.histogram2d(X[0], X[1], bins=60, range=[[-4, 4], [-3, 3]], density=True)
xc, yc = (xe[:-1] + xe[1:]) / 2, (ye[:-1] + ye[1:]) / 2
XX, YY = np.meshgrid(xc, yc, indexing="ij"); P = np.stack([XX, YY], -1)
Si = np.linalg.inv(Sigma)
f = np.exp(-0.5 * np.einsum("...i,ij,...j", P, Si, P)) / (2 * np.pi * np.sqrt(np.linalg.det(Sigma)))
print("max |histogram - density| =", np.abs(H - f).max(), " (density max", f.max(), ")")
plt.figure(figsize=(5, 4)); plt.contour(xc, yc, f.T, 6, colors="k", linewidths=0.6); plt.scatter(*X[:, :3000], s=1, alpha=0.4); plt.gca().set_aspect("equal"); plt.title("AZ with A A^T = Sigma"); plt.show()

In polar coordinates the standard Gaussian density of $\R^2$ is $\frac{e^{-r^2/2}}{2\pi}r\,dr\,d\theta$: the angle is uniform on $[0,2\pi[$, independent of the radius, and $\mathbb P(R>r)=\int_r^\infty e^{-s^2/2}s\,ds=e^{-r^2/2}$. Solving $e^{-R^2/2}=1-U_1$ gives $R=\sqrt{-2\ln(1-U_1)}$, which has the same law as $\sqrt{-2\ln U_1}$. Read backwards through the change of variables theorem, this proves that $(R\cos\Theta,R\sin\Theta)$ has density $\frac{e^{-(x^2+y^2)/2}}{2\pi}$. The tests agree, and the two coordinates are uncorrelated (in fact independent). Box–Muller is simple and exact, but numpy's ziggurat method avoids the logarithm and the trigonometric functions and is more than ten times faster here. With $A$ the Cholesky factor of $\Sigma$, $AZ$ has covariance $AA^{\mathsf T}=\Sigma$, and the histogram matches the density of Theorem 8.12 up to sampling noise of a few percent of its maximum.

### P8.3

In [ ]:
import numpy as np
rng = np.random.default_rng(2)
for d in [2, 10, 100, 1000]:
    P = rng.random((1000, d)); q = rng.random(d)
    dist = np.linalg.norm(P - q, axis=1)
    print(f"d={d:5d}: min {dist.min():7.3f}  max {dist.max():7.3f}  ratio max/min {dist.max()/dist.min():6.2f}   sqrt(d/6) = {np.sqrt(d/6):7.3f}")

For independent uniform $x,q\in[0,1]^d$, $|x-q|^2=\sum_i(x_i-q_i)^2$ is a sum of $d$ independent terms of mean $\frac16$, so by the law of large numbers $|x-q|\approx\sqrt{d/6}$, with fluctuations of order $1$ (the central limit theorem): all the distances are $\sqrt{d/6}+O(1)$, and their ratio tends to $1$. In dimension $1000$ the nearest and the farthest of $1000$ points differ by only about $14\%$. This is the same concentration as in Exercise 8.7: in high dimension, volume lives in a thin shell. For nearest-neighbour search (recommendation systems, image retrieval, $k$-NN classifiers) it means that "nearest" carries little information unless the data really lie near a low-dimensional set, which is why practical systems first reduce dimension.

### P8.4

In [ ]:
import numpy as np
from scipy.integrate import quad
f = lambda x, y: (x * x - y * y) / (x * x + y * y) ** 2
inner_y = lambda x: quad(lambda y: f(x, y), 0, 1, limit=200)[0]
inner_x = lambda y: quad(lambda x: f(x, y), 0, 1, limit=200)[0]
I1 = quad(inner_y, 0, 1, limit=200)[0]; I2 = quad(inner_x, 0, 1, limit=200)[0]
print(f"dy then dx: {I1:.6f}   dx then dy: {I2:.6f}   pi/4 = {np.pi/4:.6f}")
for eps in [0.1, 0.01, 0.001]:
    I = quad(lambda r: quad(lambda t: abs(np.cos(2 * t)) / r, 0, np.pi / 2)[0], eps, 1)[0]
    print(f"integral of |f| over the quarter annulus eps<r<1, eps={eps}: {I:.3f}")
N = 1000
a = np.eye(N) - np.eye(N, k=1)
print("truncated row sums of rows:", a.sum(axis=1)[:5], "... last:", a.sum(axis=1)[-1])
print("sum over m of (sum over n) for rows m < N:", a[:-1].sum(axis=1).sum(), "  sum over n of (sum over m):", a.sum(axis=0).sum())

The two iterated integrals come out as $\pm0.785398=\pm\frac\pi4$, and the integral of $|f|$ near the corner grows like $\ln\frac1\varepsilon$: the whole discrepancy lives at the origin, where positive mass (near the $x$-axis) and negative mass (near the $y$-axis) are both infinite. For the matrix, every complete row sums to $0$ and the column sums are $1,0,0,\dots$; in the $N\times N$ truncation the last row is cut and sums to $1$, which is where the "missing" unit sits. Truncation moves the infinite tail to the boundary, and in either example the two orders of summation disagree because the absolute series or integral is infinite, exactly the hypothesis of Fubini's theorem.

### P8.5

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
N = 2_000_000
for T in [np.array([[2.0, 1.0], [0.5, 1.5]]), np.array([[1.0, 3.0], [0.0, 1.0]]), np.array([[0.0, -1.0], [1.0, 0.0]])]:
    corners = T @ np.array([[0, 1, 0, 1], [0, 0, 1, 1]])
    lo, hi = corners.min(axis=1), corners.max(axis=1)
    P = lo[:, None] + (hi - lo)[:, None] * rng.random((2, N))
    pre = np.linalg.solve(T, P)                            # y is in T([0,1]^2) iff T^{-1} y is in [0,1]^2
    inside = np.all((pre >= 0) & (pre <= 1), axis=0)
    print(f"det {np.linalg.det(T):+.2f}: Monte Carlo area {inside.mean() * np.prod(hi - lo):.4f}")

phi = lambda x, y: (x + 0.25 * np.sin(1.5 * y), y + 0.15 * x * x)
J = lambda x, y: 1 - 0.1125 * x * np.cos(1.5 * y)
g = (np.arange(2000) + 0.5) / 2000
X, Y = np.meshgrid(g, g)
print("integral of |J| over the unit square:", J(X, Y).mean())
# Monte Carlo in the image: y is in phi([0,1]^2) iff the Newton iteration for phi(x)=y converges to a point of the square
P = rng.random((2, 400_000)) * np.array([[1.25], [1.15]])
x, y = P[0].copy(), P[1].copy()
for _ in range(30):
    u, v = phi(x, y); r1, r2 = u - P[0], v - P[1]
    a, b, c, d = 1, 0.375 * np.cos(1.5 * y), 0.3 * x, 1
    det = a * d - b * c
    x, y = x - (d * r1 - b * r2) / det, y - (-c * r1 + a * r2) / det
inside = (x >= 0) & (x <= 1) & (y >= 0) & (y <= 1)
print("Monte Carlo area of the image:", inside.mean() * 1.25 * 1.15)

The Monte Carlo areas agree with $|\det T|$ ($2.5$, $1$ and $1$) to about three decimals: shears and rotations preserve area, as Theorem 8.7 says. For the nonlinear map, $J_\varphi(x,y)=1-0.1125\,x\cos\frac{3y}2$ is positive on the square, and its integral, about $0.9626$, agrees with the Monte Carlo area of the image. Deciding whether a point lies in the image requires inverting $\varphi$, done here by Newton's method (a contraction near the solution, as in Chapter 2); the change of variables formula avoids this entirely, which is why it is the method of choice for computing volumes of images.